## Cleaning

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('01dataBaseTrainTrxRec.csv')
perfil = pd.read_csv('02dataBasePerfilRec.csv')
test_key = pd.read_csv('05dataBaseTestKeyRec.csv')

In [2]:
# --- TRAIN ---
# 1. Fechas
train['fechaOper'] = pd.to_datetime(train['fechaOper'])
train['mes'] = train['fechaOper'].dt.month
train['dia_semana'] = train['fechaOper'].dt.dayofweek
train['es_finde'] = train['dia_semana'].isin([5, 6]).astype(int)
train['es_diciembre'] = (train['mes'] == 12).astype(int)
train['es_julio'] = (train['mes'] == 7).astype(int)

In [3]:

# 2. Nulos en train
train['codGiro'] = train['codGiro'].fillna(0).astype(int)
train['ubigeoEstab'] = train['ubigeoEstab'].fillna(0).astype(int)

In [4]:

# 3. Target: log transform
train['target'] = np.log1p(train['ratingMonto'])

In [5]:
# --- PERFIL ---
# 4. saldoTcEntidad: nulo = sin tarjeta en esa entidad
for col in ['saldoTcEntidad1','saldoTcEntidad2','saldoTcEntidad3','saldoTcEntidad4']:
    perfil[col] = perfil[col].fillna('SinSaldo')

In [6]:
# 5. Rangos ordinales con nulos: rellenar con moda
for col in ['rangoIngreso', 'rangoEdad']:
    perfil[col] = perfil[col].fillna(perfil[col].mode()[0])

perfil['ubigeoCliente'] = perfil['ubigeoCliente'].fillna(perfil['ubigeoCliente'].mode()[0])

In [7]:
# 6. Encoding ordinal correcto (no LabelEncoder)
rango_map = {'Rango1':1,'Rango2':2,'Rango3':3,'Rango4':4,'Rango5':5,'Rango6':6}
saldo_map = {'SinSaldo':0,'Rango1':1,'Rango2':2,'Rango3':3,'Rango4':4,'Rango5':5,'Rango6':6}

for col in ['rangoEdad','rangoIngreso','rangoCtdProdAct','rangoCtdProdPas','rangoCtdProdSeg']:
    perfil[col] = perfil[col].map(rango_map)

for col in ['saldoTcEntidad1','saldoTcEntidad2','saldoTcEntidad3','saldoTcEntidad4']:
    perfil[col] = perfil[col].map(saldo_map)

In [8]:
# --- MERGE ---
train = train.merge(perfil, on='codCliente', how='left')
test_key = test_key.merge(perfil, on='codCliente', how='left')

print("Train limpio:", train.shape)
print("Nulos restantes train:", train.isnull().sum().sum())
print("Nulos restantes perfil:", test_key.isnull().sum().sum())

Train limpio: (1591617, 27)
Nulos restantes train: 0
Nulos restantes perfil: 0


## Feature Engineering

In [9]:
# 1. FEATURES POR CLIENTE
client_feats = train.groupby('codCliente').agg(
    # Actividad general
    total_trx_cliente        = ('ctdTrx', 'sum'),
    avg_trx_cliente          = ('ctdTrx', 'mean'),
    total_estab_visitados    = ('codEstab', 'nunique'),
    total_giros_visitados    = ('codGiro', 'nunique'),
    # Rating del cliente
    avg_rating_cliente       = ('ratingMonto', 'mean'),
    max_rating_cliente       = ('ratingMonto', 'max'),
    std_rating_cliente       = ('ratingMonto', 'std'),
    # Preferencia geográfica
    pct_lima_estab_cliente   = ('flagLimaProvEstab', 'mean'),
    # Temporalidad
    meses_activo             = ('mes', 'nunique'),
    pct_finde_cliente        = ('es_finde', 'mean'),
).reset_index()

In [10]:
# 2. FEATURES POR ESTABLECIMIENTO
estab_feats = train.groupby('codEstab').agg(
    # Popularidad
    total_clientes_estab     = ('codCliente', 'nunique'),
    total_trx_estab          = ('ctdTrx', 'sum'),
    avg_trx_estab            = ('ctdTrx', 'mean'),
    # Rating del establecimiento
    avg_rating_estab         = ('ratingMonto', 'mean'),
    max_rating_estab         = ('ratingMonto', 'max'),
    std_rating_estab         = ('ratingMonto', 'std'),
    # Temporalidad
    pct_finde_estab          = ('es_finde', 'mean'),
    meses_activo_estab       = ('mes', 'nunique'),
).reset_index()

In [11]:
# 3. FEATURES POR GIRO (rubro)
giro_feats = train.groupby('codGiro').agg(
    avg_rating_giro          = ('ratingMonto', 'mean'),
    total_clientes_giro      = ('codCliente', 'nunique'),
    total_trx_giro           = ('ctdTrx', 'sum'),
    popularidad_giro         = ('codEstab', 'nunique'),
).reset_index()

# Features cliente-giro (qué rubros prefiere cada cliente)
client_giro_feats = train.groupby(['codCliente', 'codGiro']).agg(
    trx_cliente_giro         = ('ctdTrx', 'sum'),
    avg_rating_cliente_giro  = ('ratingMonto', 'mean'),
).reset_index()

In [12]:
# Rubro favorito del cliente (el de mayor rating promedio)
giro_favorito = client_giro_feats.loc[
    client_giro_feats.groupby('codCliente')['avg_rating_cliente_giro'].idxmax()
][['codCliente', 'codGiro']].rename(columns={'codGiro': 'giro_favorito'})

client_feats = client_feats.merge(giro_favorito, on='codCliente', how='left')

In [13]:
# 4. FEATURES POR PAR CLIENTE-ESTABLECIMIENTO (interacción directa)

pair_feats = train.groupby(['codCliente', 'codEstab']).agg(
    trx_par                  = ('ctdTrx', 'sum'),
    avg_rating_par           = ('ratingMonto', 'mean'),
    max_rating_par           = ('ratingMonto', 'max'),
    visitas_par              = ('ratingMonto', 'count'),
    meses_visitado_par       = ('mes', 'nunique'),
    pct_finde_par            = ('es_finde', 'mean'),
).reset_index()

In [14]:
# 5. COLD START — establecimientos sin historial (10.7%)
# Para estos establecimientos usamos el promedio del giro al que pertenecen
# Primero necesitamos saber el giro de cada establecimiento
estab_giro = train[['codEstab','codGiro']].dropna().drop_duplicates('codEstab')
estab_feats = estab_feats.merge(estab_giro, on='codEstab', how='left')
estab_feats = estab_feats.merge(
    giro_feats[['codGiro','avg_rating_giro','total_clientes_giro']],
    on='codGiro', how='left'
)

In [15]:
# ── NUEVO: TARGET ENCODING por codGiro y ubigeoEstab ──────
giro_target = train.groupby('codGiro')['ratingMonto'].mean().reset_index()
giro_target.columns = ['codGiro', 'target_enc_giro']

ubigeo_target = train.groupby('ubigeoEstab')['ratingMonto'].mean().reset_index()
ubigeo_target.columns = ['ubigeoEstab', 'target_enc_ubigeo']

In [16]:
fecha_max = train['fechaOper'].max()
train['es_reciente'] = (train['fechaOper'] >= fecha_max - pd.DateOffset(months=3)).astype(int)

pair_reciente = train[train['es_reciente']==1].groupby(['codCliente','codEstab']).agg(
    avg_rating_par_reciente = ('ratingMonto', 'mean'),
    trx_par_reciente        = ('ctdTrx', 'sum'),
).reset_index()

In [17]:
# ── NUEVO: TENDENCIA (primera vs segunda mitad del historial) ──
train_sorted = train.sort_values(['codCliente','codEstab','fechaOper'])
train_sorted['cumidx'] = train_sorted.groupby(['codCliente','codEstab']).cumcount()
train_sorted['total_v'] = train_sorted.groupby(['codCliente','codEstab'])['codEstab'].transform('count')
train_sorted['es_segunda_mitad'] = (train_sorted['cumidx'] >= train_sorted['total_v']/2).astype(int)

tendencia = train_sorted.groupby(['codCliente','codEstab','es_segunda_mitad'])['ratingMonto'].mean().unstack()
tendencia.columns = ['rating_primera_mitad','rating_segunda_mitad']
tendencia['tendencia_rating'] = tendencia['rating_segunda_mitad'] - tendencia['rating_primera_mitad']
tendencia = tendencia.reset_index()[['codCliente','codEstab','tendencia_rating']]

In [31]:
# 6. ARMAR DATASET FINAL

def build_dataset(df, client_feats, estab_feats, pair_feats):
    df = df.merge(client_feats, on='codCliente', how='left')
    df = df.merge(estab_feats,  on='codEstab',   how='left')
    df = df.merge(pair_feats,   on=['codCliente','codEstab'], how='left')

    # Para pares sin historial directo → rellenar con promedios del establecimiento
    df['trx_par']        = df['trx_par'].fillna(0)
    df['avg_rating_par'] = df['avg_rating_par'].fillna(df['avg_rating_estab'])
    df['max_rating_par'] = df['max_rating_par'].fillna(df['avg_rating_estab'])
    df['visitas_par']    = df['visitas_par'].fillna(0)

    # Para establecimientos sin historial → usar promedio del giro
    df['avg_rating_estab'] = df['avg_rating_estab'].fillna(df['avg_rating_giro'])
    df['max_rating_estab'] = df['max_rating_estab'].fillna(df['avg_rating_giro'])
    df['total_clientes_estab'] = df['total_clientes_estab'].fillna(0)

    # Features de interacción (ratios)
    df['ratio_trx_cliente_estab'] = df['trx_par'] / (df['total_trx_cliente'] + 1)
    df['ratio_estab_sobre_cliente'] = df['total_clientes_estab'] / (df['total_estab_visitados'] + 1)
    df['diff_rating_par_vs_cliente'] = df['avg_rating_par'] - df['avg_rating_cliente']
    df['diff_rating_par_vs_estab']   = df['avg_rating_par'] - df['avg_rating_estab']

    return df

In [32]:
# Train: tiene todas las columnas originales
X_train = build_dataset(train, client_feats, estab_feats, pair_feats)

# Test: solo tiene codCliente y codEstab + perfil (ya mergeado antes)
X_test  = build_dataset(test_key, client_feats, estab_feats, pair_feats)


In [33]:
drop_cols = [
    'codCliente', 'codEstab',
    'ratingMonto', 'target',
    'fechaOper', 'mes', 'dia_semana', 'es_finde', 'es_diciembre', 'es_julio',
    'codGiro_x', 'codGiro_y',
    'es_reciente',        # ← NUEVO: columna auxiliar
    'cumidx', 'total_v', 'es_segunda_mitad',  # ← NUEVO: auxiliares de tendencia
]

# Features = columnas que están en X_test (fuente de verdad)
features = [c for c in X_test.columns if c not in drop_cols]

# Verificar que todas las features existen también en X_train
features = [c for c in features if c in X_train.columns]

y_train = X_train['target']  # log1p(ratingMonto)

print(f"Features totales: {len(features)}")
print(f"Shape X_train: {X_train[features].shape}")
print(f"Shape X_test:  {X_test[features].shape}")
print(f"\nNulos en X_train: {X_train[features].isnull().sum().sum()}")
print(f"Nulos en X_test:  {X_test[features].isnull().sum().sum()}")
print("\nLista de features:")
for f in features:
    print(f"  - {f}")

Features totales: 44
Shape X_train: (1591617, 44)
Shape X_test:  (467203, 44)

Nulos en X_train: 22213
Nulos en X_test:  670914

Lista de features:
  - rangoEdad
  - rangoIngreso
  - flagGenero
  - flagLimaProvCliente
  - ubigeoCliente
  - rangoCtdProdAct
  - rangoCtdProdPas
  - rangoCtdProdSeg
  - flagBxi
  - saldoTcEntidad1
  - saldoTcEntidad2
  - saldoTcEntidad3
  - saldoTcEntidad4
  - total_trx_cliente
  - avg_trx_cliente
  - total_estab_visitados
  - total_giros_visitados
  - avg_rating_cliente
  - max_rating_cliente
  - std_rating_cliente
  - pct_lima_estab_cliente
  - meses_activo
  - pct_finde_cliente
  - giro_favorito
  - total_clientes_estab
  - total_trx_estab
  - avg_trx_estab
  - avg_rating_estab
  - max_rating_estab
  - std_rating_estab
  - pct_finde_estab
  - meses_activo_estab
  - avg_rating_giro
  - total_clientes_giro
  - trx_par
  - avg_rating_par
  - max_rating_par
  - visitas_par
  - meses_visitado_par
  - pct_finde_par
  - ratio_trx_cliente_estab
  - ratio_estab_s

In [34]:
print(f"Nulos en X_train: {X_train[features].isnull().sum().sum()}")
print(f"Nulos en X_test:  {X_test[features].isnull().sum().sum()}")

Nulos en X_train: 22213
Nulos en X_test:  670914


In [35]:
print("Nulos restantes en X_test:")
print(X_test[features].isnull().sum()[X_test[features].isnull().sum() > 0])

Nulos restantes en X_test:
std_rating_cliente                 5
total_trx_estab                 8993
avg_trx_estab                   8993
avg_rating_estab                8993
max_rating_estab                8993
std_rating_estab               16939
pct_finde_estab                 8993
meses_activo_estab              8993
avg_rating_giro                 8993
total_clientes_giro             8993
avg_rating_par                  8993
max_rating_par                  8993
meses_visitado_par            273027
pct_finde_par                 273027
diff_rating_par_vs_cliente      8993
diff_rating_par_vs_estab        8993
dtype: int64


In [ ]:
X_test

In [ ]:
X_train

## Model

In [36]:
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# ============================================================
# MODELO — LightGBM con validación cruzada 5 folds
# ============================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds  = np.zeros(len(X_train))
feat_importance = np.zeros(len(features))

params = {
    'n_estimators'     : 500,
    'learning_rate'    : 0.03,
    'num_leaves'       : 63,       # reducir complejidad
    'min_child_samples': 20,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,
    'min_gain_to_split': 0.01,     # evitar splits sin ganancia
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,       # silenciar warnings
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f'\n===== Fold {fold+1} =====')

    X_tr  = X_train[features].iloc[tr_idx]
    X_val = X_train[features].iloc[val_idx]
    y_tr  = y_train.iloc[tr_idx]
    y_val = y_train.iloc[val_idx]

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(100)
        ]
    )

    oof_preds[val_idx] = model.predict(X_val)
    feat_importance += model.feature_importances_ / 5

# ============================================================
# MÉTRICA GLOBAL
# ============================================================
oof_rmse = np.sqrt(mean_squared_error(y_train, oof_preds))
print(f'\n✅ OOF RMSE (log scale): {oof_rmse:.6f}')

# Convertir de vuelta a escala original
oof_real      = np.expm1(oof_preds)
y_real        = np.expm1(y_train)
oof_rmse_real = np.sqrt(mean_squared_error(y_real, oof_real))
print(f'✅ OOF RMSE (escala original): {oof_rmse_real:.6f}')

# ============================================================
# FEATURE IMPORTANCE
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

fi_df = pd.DataFrame({
    'feature'   : features,
    'importance': feat_importance
}).sort_values('importance', ascending=False)

print('\nTop 15 features más importantes:')
print(fi_df.head(15).to_string(index=False))

plt.figure(figsize=(10, 8))
fi_df.head(20).plot(kind='barh', x='feature', y='importance',figsize=(10, 8), color='steelblue', legend=False)
plt.title('Feature Importance — LightGBM')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


===== Fold 1 =====
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 7.98321e-05
[200]	valid_0's l2: 7.81734e-05
Early stopping, best iteration is:
[203]	valid_0's l2: 7.81725e-05

===== Fold 2 =====
Training until validation scores don't improve for 50 rounds


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np

# 1. Leer archivos de test
test    = pd.read_csv('03dataBaseTestRec.csv')
test_key = pd.read_csv('05dataBaseTestKeyRec.csv')

# 2. Preparar X_test con el mismo pipeline
test_key = test_key.merge(perfil, on='codCliente', how='left')
X_test   = build_dataset(test_key, client_feats, estab_feats, pair_feats)

# Resolver nulos restantes del cold start
X_test['avg_rating_par'] = X_test['avg_rating_par'].fillna(X_test['avg_rating_estab'])
X_test['max_rating_par'] = X_test['max_rating_par'].fillna(X_test['avg_rating_estab'])
X_test['diff_rating_par_vs_cliente'] = X_test['avg_rating_par'] - X_test['avg_rating_cliente']
X_test['diff_rating_par_vs_estab']   = X_test['avg_rating_par'] - X_test['avg_rating_estab']

print(f"Nulos en X_test: {X_test[features].isnull().sum().sum()}")

# 3. Reentrenar modelo final con TODOS los datos de train
print("\nEntrenando modelo final con todo el train...")
model_final = lgb.LGBMRegressor(**params)
model_final.fit(
    X_train[features], y_train,
    callbacks=[lgb.log_evaluation(100)]
)

# 4. Predecir
test_preds = model_final.predict(X_test[features])
test_preds = np.expm1(test_preds)          # revertir log1p
test_preds = np.clip(test_preds, 0, 1)     # ratingMonto debe estar entre 0 y 1

# 5. Generar submission en formato del 03
test['ratingMonto'] = test_preds
test.to_csv('submission.csv', index=False)

print(f"\n✅ Submission generado: {len(test)} filas")
print(test.head(10))